In [0]:
import dlt
from pyspark.sql.functions import current_timestamp

@dlt.table(
    name="bronze_fraud_dlt",
    comment="Raw credit card fraud data ingested from ADLS2",
    table_properties={"quality": "bronze"}
)
def bronze_fraud_dlt():
    storage_account_name = "shruthistorageaccount"
    bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
    
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(bronze_path + "credit_card_fraud.csv")
        .withColumn("ingested_at", current_timestamp())
    )

    dlt.read("bronze_fraud_dlt").show()

In [0]:
@dlt.table(
    name="silver_fraud_dlt",
    comment="Cleaned and transformed fraud transactions",
    table_properties={"quality": "silver"}
)
@dlt.expect("valid_transaction_id", "transaction_id IS NOT NULL")
@dlt.expect("valid_amount", "amount > 0")
@dlt.expect_or_drop("valid_fraud_flag", "IsFraud IN (0, 1)")
def silver_fraud_dlt():
    from pyspark.sql.functions import col, month, year
    
    return (
        dlt.read("bronze_fraud_dlt")
        .withColumnRenamed("TransactionID", "transaction_id")
        .withColumnRenamed("TransactionDate", "transaction_date")
        .withColumnRenamed("Amount", "amount")
        .withColumnRenamed("MerchantID", "merchant_id")
        .withColumnRenamed("TransactionType", "transaction_type")
        .withColumnRenamed("Location", "location")
        .withColumnRenamed("IsFraud", "is_fraud")
        .withColumn("transaction_month", month(col("transaction_date")))
        .withColumn("transaction_year", year(col("transaction_date")))
    )

In [0]:
@dlt.table(
    name="gold_fraud_by_location_dlt",
    comment="Fraud aggregations by location",
    table_properties={"quality": "gold"}
)
def gold_fraud_by_location_dlt():
    from pyspark.sql.functions import count, sum, when, round, col
    
    return (
        dlt.read("silver_fraud_dlt")
        .groupBy("location")
        .agg(
            count("transaction_id").alias("total_transactions"),
            sum(when(col("is_fraud") == 1, 1).otherwise(0)).alias("fraud_count"),
            round(sum("amount"), 2).alias("total_amount")
        )
    )